# ML + Qlib 风格特征工程 + 回测 完整流程

本 Notebook 展示如何：
1. 使用 Qlib 风格的 Alpha 特征工程
2. 训练 ML 模型（LightGBM/XGBoost/神经网络）
3. 使用 Backtrader 进行策略回测

流程：数据加载 → Qlib 特征工程 → 时序划分 → ML 训练 → 回测

## 1. 导入模块

In [6]:
import sys
import os
from pathlib import Path

# 设置项目路径
PROJECT_ROOT = Path(os.getcwd()).parent if 'quant-learning' in str(Path.cwd()) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

# 核心模块
from data_loader import StockDataLoader, time_series_split
from features.qlib_features import QlibFeatureEngineer, QlibFeatureEngineerV2, create_alpha_features
from config import DATA_PATH
from models.sklearn_models import (
    RidgeRegressionModel, RandomForestModel, XGBoostModel, LightGBMModel
)
from models.pytorch_models import MLPModel, LSTMModel

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 所有模块导入成功！')
print(f'数据路径: {DATA_PATH}')

✅ 所有模块导入成功！
数据路径: /Users/harryyfhu/quant-learning/data


## 2. 加载数据

In [7]:
# 加载数据
loader = StockDataLoader(DATA_PATH / 'a_stock_history_price_20260223.csv')
df = loader.load(years_back=None)

# 可选：限制股票数量以加速
selected_codes = loader.select_sample_codes(n=200)
df = df[df['code'].isin(selected_codes)]

print(f"\n📊 数据集信息:")
print(f"   总记录数: {len(df):,}")
print(f"   股票数量: {df['code'].nunique()}")
print(f"   日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")

# 查看数据样例
df.head()

📂 加载数据: /Users/harryyfhu/quant-learning/data/a_stock_history_price_20260223.csv
   原始: 7,783,131 行

   原始列名: ['date', 'stock_code', 'name', 'open', 'close', 'high', 'low', 'price_change', 'price_change_rate', 'volume', 'turnover_rate', 'market_cap', 'dividends', 'stock_splits']

   各列取值样例（前3行）:
         date  stock_code  name   open  close   high    low  price_change  price_change_rate       volume  turnover_rate    market_cap  dividends  stock_splits
0  2020-01-02           1  平安银行  13.74  13.93  13.99  13.66           NaN                NaN  153023187.0           0.79  2.703244e+11        0.0           0.0
1  2020-01-03           1  平安银行  13.98  14.18  14.29  13.97          0.25               1.79  111619481.0           0.58  2.751759e+11        0.0           0.0
2  2020-01-06           1  平安银行  14.04  14.09  14.31  13.96         -0.09              -0.63   86208350.0           0.44  2.734294e+11        0.0           0.0

   各列数据类型:
date                  object
stock_code             

,date,code,name,open,close,high,low,price_change,price_change_rate,volume,turnover_rate,market_cap,dividends,stock_splits,amount
34155,2020-01-02,000034,神州数码,18.56,19.15,19.31,18.48,NaN,NaN,17321895.0,2.39,1.385396e+10,0.0,0.0,3.317143e+08
34156,2020-01-03,000034,神州数码,19.20,19.16,19.76,18.91,0.01,0.05,21744199.0,3.01,1.386119e+10,0.0,0.0,4.166189e+08
34157,2020-01-06,000034,神州数码,19.12,18.73,19.27,18.51,-0.43,-2.24,22846719.0,3.16,1.355011e+10,0.0,0.0,4.279190e+08
34158,2020-01-07,000034,神州数码,18.60,19.38,19.39,18.58,0.65,3.47,21814889.0,3.02,1.402035e+10,0.0,0.0,4.227725e+08
34159,2020-01-08,000034,神州数码,19.12,18.48,19.19,18.43,-0.90,-4.64,21716428.0,3.00,1.336925e+10,0.0,0.0,4.013196e+08


## 3. 时序数据集划分

⚠️ **重要**: 必须按时间划分，防止数据泄露！

In [8]:
# 时间序列划分
train_df, val_df, test_df = time_series_split(
    df, 
    train_ratio=0.6,   # 60% 训练
    val_ratio=0.2      # 20% 验证，20% 测试
)

# 验证无时间重叠
print("\n🔒 时序划分验证:")
print(f"   训练集: {train_df['date'].min().date()} ~ {train_df['date'].max().date()}")
print(f"   验证集: {val_df['date'].min().date()} ~ {val_df['date'].max().date()}")
print(f"   测试集: {test_df['date'].min().date()} ~ {test_df['date'].max().date()}")

# 确认无重叠
assert train_df['date'].max() < val_df['date'].min(), "❌ 训练集和验证集重叠"
assert val_df['date'].max() < test_df['date'].min(), "❌ 验证集和测试集重叠"
print("   ✅ 无时间穿越风险")


⏰ 时间序列数据集划分...
   训练集: 130,724 (2020-01-02 ~ 2023-09-01)
   验证集: 54,951 (2023-09-04 ~ 2024-11-27)
   测试集: 58,045 (2024-11-28 ~ 2026-02-13)

🔒 时序划分验证:
   训练集: 2020-01-02 ~ 2023-09-01
   验证集: 2023-09-04 ~ 2024-11-27
   测试集: 2024-11-28 ~ 2026-02-13
   ✅ 无时间穿越风险


## 4. Qlib 风格特征工程

使用类似 Qlib 的 Alpha 表达式系统构建特征

In [9]:
# 创建 Qlib 特征工程器
engineer = QlibFeatureEngineer()
engineer.build_alpha_features()  # 构建 30 个基础 Alpha 特征

print("\n" + "="*60)
print("🔧 步骤1: 训练集特征工程（仅在训练集上计算）")
print("="*60)
train_features = engineer.transform(train_df)

print("\n🔧 步骤2: 验证集特征工程")
val_features = engineer.transform(val_df)

print("\n🔧 步骤3: 测试集特征工程")
test_features = engineer.transform(test_df)

# 获取特征列（使用工程器的特征名称，不再依赖 alpha_ 前缀）
alpha_cols = engineer.feature_names
print(f"\n📋 特征列表 ({len(alpha_cols)} 个):")
for i, col in enumerate(alpha_cols, 1):
    print(f"   {i:2d}. {col}")


✅ 已构建 30 个 Alpha 特征

🔧 步骤1: 训练集特征工程（仅在训练集上计算）
🔧 计算基础特征...
🔧 计算 30 个 Alpha 特征...
✅ 特征计算完成，共 30 个特征

🔧 步骤2: 验证集特征工程
🔧 计算基础特征...
🔧 计算 30 个 Alpha 特征...
✅ 特征计算完成，共 30 个特征

🔧 步骤3: 测试集特征工程
🔧 计算基础特征...
🔧 计算 30 个 Alpha 特征...
✅ 特征计算完成，共 30 个特征

📋 特征列表 (30 个):
    1. returns_5d
    2. returns_10d
    3. returns_20d
    4. returns_60d
    5. close_ma5_ratio
    6. close_ma20_ratio
    7. ma5_ma20_ratio
    8. close_std_20d
    9. ret1_std_20d
   10. vol_ma5_ma20_ratio
   11. vol_returns_5d
   12. price_pos_20d
   13. close_delta_5d
   14. close_delta_10d
   15. close_dev_ma20
   16. amplitude_mean_20d
   17. pv_correlation
   18. ret1_rank
   19. ret5_rank
   20. vol_rank
   21. rsi_proxy_5d
   22. close_acceleration
   23. ret5_vol20_cross
   24. pos20_volret5_cross
   25. close_rank_pos_20d
   26. trend_ratio_5_20
   27. open_close_ratio
   28. intraday_return
   29. close_hl_pos
   30. ret5_std5_ratio


## 5. 添加目标变量（标签）

预测未来 N 日的收益率

In [10]:
def add_target_variable(df: pd.DataFrame, pred_horizon: int = 5) -> pd.DataFrame:
    """
    添加目标变量：未来 N 日收益率
    """
    df = df.copy()
    
    # 计算未来收益率
    df[f'target_return_{pred_horizon}d'] = df.groupby('code')['close'].shift(-pred_horizon) / df['close'] - 1
    
    # 删除无法计算目标的行
    df = df.dropna(subset=[f'target_return_{pred_horizon}d'])
    
    return df

# 从 config 导入预测周期参数（统一管理，不在此处硬编码）
from config import PRED_HORIZON

print(f"📊 添加目标变量（预测未来 {PRED_HORIZON} 日收益率）...")
train_features = add_target_variable(train_features, PRED_HORIZON)
val_features = add_target_variable(val_features, PRED_HORIZON)
test_features = add_target_variable(test_features, PRED_HORIZON)

target_col = f'target_return_{PRED_HORIZON}d'
print(f"\n目标变量统计:")
print(f"   训练集: {train_features[target_col].describe().round(4).to_dict()}")


📊 添加目标变量（预测未来 5 日收益率）...

目标变量统计:
   训练集: {'count': 129819.0, 'mean': 0.0023, 'std': 0.0687, 'min': -0.4858, '25%': -0.0324, '50%': -0.0014, '75%': 0.0296, 'max': 1.4925}


## 6. 数据标准化

⚠️ **重要**: 只在训练集上 fit，验证集和测试集只 transform

In [ ]:
# 数据清洗：删除包含 NaN 的行print("🔧 数据清洗...")# 检查 NaN 数量train_nan = train_features[alpha_cols + [target_col]].isna().sum().sum()val_nan = val_features[alpha_cols + [target_col]].isna().sum().sum()test_nan = test_features[alpha_cols + [target_col]].isna().sum().sum()print(f"   训练集 NaN 数量: {train_nan}")print(f"   验证集 NaN 数量: {val_nan}")print(f"   测试集 NaN 数量: {test_nan}")# 删除 NaNtrain_clean = train_features.dropna(subset=alpha_cols + [target_col])val_clean = val_features.dropna(subset=alpha_cols + [target_col])test_clean = test_features.dropna(subset=alpha_cols + [target_col])print(f"\n   清洗后数据量:")print(f"   训练集: {len(train_clean):,} (删除 {len(train_features) - len(train_clean):,} 行)")print(f"   验证集: {len(val_clean):,} (删除 {len(val_features) - len(val_clean):,} 行)")print(f"   测试集: {len(test_clean):,} (删除 {len(test_features) - len(test_clean):,} 行)")# 准备数据scaler = StandardScaler()# 提取特征和目标X_train = train_clean[alpha_cols].valuesy_train = train_clean[target_col].valuesX_val = val_clean[alpha_cols].valuesy_val = val_clean[target_col].valuesX_test = test_clean[alpha_cols].valuesy_test = test_clean[target_col].values# 标准化（只在训练集上 fit）print("\n📊 数据标准化...")X_train_scaled = scaler.fit_transform(X_train)X_val_scaled = scaler.transform(X_val)X_test_scaled = scaler.transform(X_test)print(f"\n✅ 数据准备完成:")print(f"   训练集: X{X_train_scaled.shape}, y{y_train.shape}")print(f"   验证集: X{X_val_scaled.shape}, y{y_val.shape}")print(f"   测试集: X{X_test_scaled.shape}, y{y_test.shape}")print(f"\n📊 目标变量均值:")print(f"   训练集: {y_train.mean():.6f}")print(f"   验证集: {y_val.mean():.6f}")print(f"   测试集: {y_test.mean():.6f}")

## 7. 训练 ML 模型

对比多个模型：
- Ridge 回归（线性基准）
- LightGBM（梯度提升）
- XGBoost（梯度提升）
- MLP（神经网络，可选）

In [11]:
# 存储所有模型结果
models_results = {}

print("\n" + "="*60)
print("🤖 模型 1: Ridge 回归（线性基准）")
print("="*60)

model_ridge = RidgeRegressionModel(alpha=1.0)
history_ridge = model_ridge.fit(X_train_scaled, y_train, X_val_scaled, y_val)
metrics_ridge = model_ridge.evaluate(X_test_scaled, y_test)

models_results['Ridge'] = {'model': model_ridge, 'metrics': metrics_ridge}
print(f"\nRidge 测试集表现:")
for k, v in metrics_ridge.items():
    print(f"   {k}: {v:.4f}")


🤖 模型 1: Ridge 回归（线性基准）


NameError: name 'X_train_scaled' is not defined

In [ ]:
print("\n" + "="*60)
print("🤖 模型 2: LightGBM")
print("="*60)

model_lgb = LightGBMModel(
    n_estimators=200,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

history_lgb = model_lgb.fit(X_train_scaled, y_train, X_val_scaled, y_val)
metrics_lgb = model_lgb.evaluate(X_test_scaled, y_test)

models_results['LightGBM'] = {'model': model_lgb, 'metrics': metrics_lgb}
print(f"\nLightGBM 测试集表现:")
for k, v in metrics_lgb.items():
    print(f"   {k}: {v:.4f}")

In [ ]:
print("\n" + "="*60)
print("🤖 模型 3: XGBoost")
print("="*60)

model_xgb = XGBoostModel(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

history_xgb = model_xgb.fit(X_train_scaled, y_train, X_val_scaled, y_val)
metrics_xgb = model_xgb.evaluate(X_test_scaled, y_test)

models_results['XGBoost'] = {'model': model_xgb, 'metrics': metrics_xgb}
print(f"\nXGBoost 测试集表现:")
for k, v in metrics_xgb.items():
    print(f"   {k}: {v:.4f}")

In [ ]:
# 可选：MLP 神经网络（需要 torch）
try:
    print("\n" + "="*60)
    print("🤖 模型 4: MLP 神经网络")
    print("="*60)
    
    model_mlp = MLPModel(
        input_dim=X_train_scaled.shape[1],
        hidden_dims=[128, 64, 32],
        dropout_rate=0.3,
        epochs=50,
        batch_size=512,
        lr=0.001,
        early_stopping_patience=10
    )
    
    history_mlp = model_mlp.fit(X_train_scaled, y_train, X_val_scaled, y_val)
    metrics_mlp = model_mlp.evaluate(X_test_scaled, y_test)
    
    models_results['MLP'] = {'model': model_mlp, 'metrics': metrics_mlp}
    print(f"\nMLP 测试集表现:")
    for k, v in metrics_mlp.items():
        print(f"   {k}: {v:.4f}")
except ImportError as e:
    print(f"⚠️ PyTorch 未安装，跳过 MLP: {e}")

## 8. 模型对比与选择

In [ ]:
# 模型对比
print("\n" + "="*60)
print("📊 模型对比（测试集）")
print("="*60)

comparison_df = pd.DataFrame(
    {name: result['metrics'] for name, result in models_results.items()}
).T

print(comparison_df.round(4))

# 选择最佳模型（根据 IC 或 R2）
best_model_name = comparison_df['R2'].idxmax()
print(f"\n🏆 最佳模型: {best_model_name}")
print(f"   R2: {comparison_df.loc[best_model_name, 'R2']:.4f}")
print(f"   RMSE: {comparison_df.loc[best_model_name, 'RMSE']:.4f}")

best_model = models_results[best_model_name]['model']

## 9. 生成预测结果

In [ ]:
# 使用最佳模型生成预测print(f"\n🔮 使用 {best_model_name} 生成预测...")# 测试集预测y_pred_test = best_model.predict(X_test_scaled)test_clean['pred_return'] = y_pred_test# 验证集预测（用于分析）y_pred_val = best_model.predict(X_val_scaled)val_clean['pred_return'] = y_pred_val# 查看预测分布print("\n📊 预测值分布:")print(f"   验证集: mean={y_pred_val.mean():.4f}, std={y_pred_val.std():.4f}")print(f"   测试集: mean={y_pred_test.mean():.4f}, std={y_pred_test.std():.4f}")# 预测 vs 实际散点图fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].scatter(y_val, y_pred_val, alpha=0.3, s=1)axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)axes[0].set_xlabel('Actual Return')axes[0].set_ylabel('Predicted Return')axes[0].set_title(f'{best_model_name} - Validation Set')axes[1].scatter(y_test, y_pred_test, alpha=0.3, s=1)axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)axes[1].set_xlabel('Actual Return')axes[1].set_ylabel('Predicted Return')axes[1].set_title(f'{best_model_name} - Test Set')plt.tight_layout()plt.savefig('./prediction_scatter.png', dpi=150)plt.show()print("\n✅ 预测散点图已保存: ./prediction_scatter.png")

## 10. 特征重要性分析

In [ ]:
# 获取特征重要性（仅树模型支持）
if hasattr(best_model.model, 'feature_importances_'):
    importances = best_model.model.feature_importances_
    
    # 创建重要性 DataFrame
    importance_df = pd.DataFrame({
        'feature': alpha_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("\n📊 Top 15 重要特征:")
    print(importance_df.head(15).to_string(index=False))
    
    # 可视化
    plt.figure(figsize=(10, 8))
    top_n = 20
    plt.barh(range(top_n), importance_df['importance'].head(top_n))
    plt.yticks(range(top_n), importance_df['feature'].head(top_n))
    plt.xlabel('Importance')
    plt.title(f'{best_model_name} - Feature Importance (Top {top_n})')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('./feature_importance.png', dpi=150)
    plt.show()
    print("\n✅ 特征重要性图已保存: ./feature_importance.png")
else:
    print("⚠️ 当前模型不支持特征重要性分析")

## 11. 回测 - TopK 策略

每天选择预测收益最高的 K 只股票进行等权重投资

In [ ]:
# 导入回测模块
import importlib
import strategy.backtrader_topk_strategy as backtrader_topk_strategy
importlib.reload(backtrader_topk_strategy)
from strategy.backtrader_topk_strategy import run_topk_backtest_ultra_optimized, create_trade_chart

# 从 config 导入回测参数（统一管理，不在此处硬编码）
from config import (
    TOP_K_STOCKS as TOP_K,
    BACKTEST_REBALANCE_FREQ as REBALANCE_FREQ,
    BACKTEST_INITIAL_CASH as INITIAL_CASH,
    BACKTEST_COMMISSION as COMMISSION,
)

print("\n" + "="*60)
print("🚀 TopK 策略回测")
print("="*60)

print(f"\n📋 回测配置:")
print(f"   选股数量 (TopK): {TOP_K}")
print(f"   再平衡频率: 每 {REBALANCE_FREQ} 个交易日")
print(f"   初始资金: {INITIAL_CASH:,.0f}")
print(f"   手续费率: {COMMISSION*100:.3f}%")

# 运行回测
result_topk = run_topk_backtest_ultra_optimized(
    test_df=test_clean,
    pred_col='pred_return',
    code_col='code',
    date_col='date',
    price_col='close',
    top_k=TOP_K,
    rebalance_freq=REBALANCE_FREQ,
    equal_weight=True,
    min_data_days=150,
    initial_cash=INITIAL_CASH,
    commission=COMMISSION
)

print(f"\n📊 回测结果:")
print(f"   总收益率: {result_topk['total_return']*100:.2f}%")
print(f"   年化收益率: {result_topk.get('annual_return', 0)*100:.2f}%")
print(f"   最大回撤: {result_topk.get('max_drawdown', 0)*100:.2f}%")
print(f"   夏普比率: {result_topk.get('sharpe_ratio', 0):.3f}")
print(f"   总耗时: {result_topk.get('elapsed_time', 0):.2f}s")

# 绘制回测图表
create_trade_chart(result_topk, title=f"TopK Strategy (K={TOP_K})", save_path="./topk_result.png")
print("\n✅ 回测图表已保存: ./topk_result.png")


## 12. 回测 - Threshold 策略

根据预测收益阈值进行买入/卖出决策

In [ ]:
import strategy.backtrader_threshold_strategy as backtrader_threshold_strategy
importlib.reload(backtrader_threshold_strategy)
from strategy.backtrader_threshold_strategy import (
    run_threshold_backtest_ultra,
    create_trade_chart as create_threshold_chart,
    analyze_trade_profit_loss
)

# 从 config 导入阈值策略参数
from config import (
    THRESHOLD_BUY as BUY_THRESHOLD,
    THRESHOLD_SELL as SELL_THRESHOLD,
    THRESHOLD_MAX_POSITIONS as MAX_POSITIONS,
)

print("\n" + "="*60)
print("🚀 Threshold 策略回测")
print("="*60)

print(f"\n📋 回测配置:")
print(f"   买入阈值: {BUY_THRESHOLD*100:.1f}%")
print(f"   卖出阈值: {SELL_THRESHOLD*100:.1f}%")
print(f"   最大持仓: {MAX_POSITIONS} 只")
print(f"   再平衡频率: 每 {REBALANCE_FREQ} 个交易日")

# 运行回测（只使用主板股票）
test_features_main = test_clean[
    test_clean['code'].astype(str).str.match(r'^(600|601|603|605|000|001|002|003)\d{3}$')
].copy()

result_threshold = run_threshold_backtest_ultra(
    test_df=test_features_main,
    pred_col='pred_return',
    buy_threshold=BUY_THRESHOLD,
    sell_threshold=SELL_THRESHOLD,
    max_positions=MAX_POSITIONS,
    rebalance_freq=REBALANCE_FREQ,
    min_data_days=150,
    initial_cash=INITIAL_CASH,
    commission=COMMISSION,
    print_log=False
)

print(f"\n📊 Threshold 策略回测结果:")
print(f"   总收益率: {result_threshold['total_return']*100:.2f}%")
print(f"   年化收益率: {result_threshold.get('annual_return', 0)*100:.2f}%")
print(f"   最大回撤: {result_threshold.get('max_drawdown', 0)*100:.2f}%")
print(f"   夏普比率: {result_threshold.get('sharpe_ratio', 0):.3f}")
print(f"   总耗时: {result_threshold.get('elapsed_time', 0):.2f}s")

# 绘制回测图表
create_threshold_chart(result_threshold, title="Threshold Strategy", save_path="./threshold_result.png")
print("\n✅ Threshold 回测图表已保存: ./threshold_result.png")


## 13. 交易盈亏分析

In [ ]:
# 分析 Threshold 策略的交易明细
if 'trades' in result_threshold and len(result_threshold['trades']) > 0:
    print("\n" + "="*60)
    print("📊 Threshold 策略交易盈亏分析")
    print("="*60)
    
    analyze_trade_profit_loss(result_threshold, save_path='./threshold_pnl.png')
    print("\n✅ 盈亏分析图已保存: ./threshold_pnl.png")
else:
    print("⚠️ 没有交易记录可供分析")

## 14. 策略对比总结

In [ ]:
# 策略对比
print("\n" + "="*60)
print("📊 策略对比总结")
print("="*60)

# 提取关键指标
strategies = {
    'TopK': result_topk,
    'Threshold': result_threshold
}

comparison_data = []
for name, result in strategies.items():
    comparison_data.append({
        '策略': name,
        '总收益率(%)': f"{result['total_return']*100:.2f}",
        '年化收益率(%)': f"{result.get('annual_return', 0)*100:.2f}",
        '最大回撤(%)': f"{result.get('max_drawdown', 0)*100:.2f}",
        '夏普比率': f"{result.get('sharpe_ratio', 0):.3f}",
        '交易次数': len(result.get('trades', [])),
        '回测时间(s)': f"{result.get('elapsed_time', 0):.2f}"
    })

comparison_result = pd.DataFrame(comparison_data)
print(comparison_result.to_string(index=False))

# 找出最佳策略
best_strategy = max(strategies.items(), key=lambda x: x[1]['total_return'])
print(f"\n🏆 最佳策略: {best_strategy[0]}")
print(f"   总收益率: {best_strategy[1]['total_return']*100:.2f}%")

## 15. 模型与策略参数调优建议

基于当前结果，可以考虑以下优化方向：

In [ ]:
print("\n" + "="*60)
print("💡 优化建议")
print("="*60)

suggestions = """
1. 特征工程优化：
   - 尝试 QlibFeatureEngineerV2 的高级特征
   - 添加行业/板块特征
   - 添加宏观因子（市场波动率、利率等）

2. 模型优化：
   - 使用交叉验证选择超参数
   - 尝试集成学习（Stacking/Boosting）
   - 添加特征选择（移除低重要性特征）

3. 策略优化：
   - 调整 TopK 的 K 值
   - 优化买入/卖出阈值
   - 添加止损/止盈逻辑
   - 考虑仓位管理（Kelly 公式等）

4. 风险管理：
   - 添加最大持仓限制
   - 添加行业分散度约束
   - 考虑市场状态过滤（只在趋势明确时交易）
"""

print(suggestions)

# 保存结果摘要
summary = f"""
# ML + Qlib 特征工程回测结果摘要

## 模型性能（测试集）
{comparison_df.round(4).to_string()}

## 最佳模型
- 模型: {best_model_name}
- R2: {comparison_df.loc[best_model_name, 'R2']:.4f}
- RMSE: {comparison_df.loc[best_model_name, 'RMSE']:.4f}

## 策略回测结果
{comparison_result.to_string(index=False)}

## 回测配置
- 预测周期: {PRED_HORIZON} 日
- 训练集: {train_df['date'].min().date()} ~ {train_df['date'].max().date()}
- 验证集: {val_df['date'].min().date()} ~ {val_df['date'].max().date()}
- 测试集: {test_df['date'].min().date()} ~ {test_df['date'].max().date()}
- 特征数量: {len(alpha_cols)}
"""

with open('./backtest_summary.md', 'w') as f:
    f.write(summary)

print("\n✅ 结果摘要已保存: ./backtest_summary.md")
print("\n🎉 全部流程完成！")

---

## 附录：使用 QlibFeatureEngineerV2（高级特征）

如果需要更多特征，可以切换到 V2 版本：

In [ ]:
# 使用高级特征（可选）
# engineer_v2 = QlibFeatureEngineerV2()
# engineer_v2.build_alpha_features().build_advanced_features()
# train_features_v2 = engineer_v2.transform(train_df)
# ...（后续流程相同）